In [ ]:
import ipywidgets as widgets
from ipyleaflet import Map, DrawControl, Polygon, GeoData
from fetchez.registry import ModuleRegistry, BundleRegistry
import os
import threading
import geopandas as gpd
import logging
import fetchez.recipe
import fetchez.core

shared_layout = widgets.Layout(height="auto")

# --- Header Block ---
header = widgets.HTML("<h2>🌐 Globato DEM Builder</h2><hr/>")

# --- Load Registries & Extract Descriptions ---
ModuleRegistry.load_all()
BundleRegistry.load_all()
registry = ModuleRegistry.get_registry()
registry.update(BundleRegistry.get_registry())

globato_sources = {}
for name, meta in sorted(registry.items()):
    # Filter for modules/bundles tagged with 'glob-stream'
    if "glob-stream" in meta.get("tags", []) and name not in meta.get("aliases", []):
        # Extract the description to use for our hover tooltip
        desc = meta.get("description") or meta.get("desc", "No description provided.")
        globato_sources[name] = desc.strip().split("\n")[0]

# --- Left Column: Map Selector ---
m = Map(center=[44.6, -124.05], zoom=10, layout=shared_layout)
draw_control = DrawControl(rectangle={"shapeOptions": {"color": "#0074D9"}})
m.add(draw_control)

session_layers = []

# --- Right Column: Form Options ---
# Group A: Processing Parameters
region_input = widgets.Text(description="Region:", placeholder="Auto-populated by map")
upload_widget = widgets.FileUpload(
    accept=".geojson,.gpkg",  # Stick to single-file vector formats for simplicity
    multiple=False,
    description="Upload Vector Region",
    layout=widgets.Layout(width="100%"),
)
increment_input = widgets.Dropdown(
    options=["1s", "1/3s", "1/9s", "3s", "30m"], value="1s", description="Increment:"
)
srs_input = widgets.Dropdown(
    options=["EPSG:4326+3855", "EPSG:4269+5703", "EPSG:3857"],
    value="EPSG:4326+3855",
    description="Target SRS:",
)
# other_text = widgets.Text(
#     placeholder='Type your custom SRS information here...',
#     description='Specify:',
#     layout=widgets.Layout(display='none') # Hidden by default#
# )


# def handle_dropdown_change(change):
#     if change['new'] == 'Other':
#         other_text.layout.display = 'block'   # Show text box
#     else:
#         other_text.layout.display = 'none'    # Hide text box
#         other_text.value = ''                 # Reset text box value

# srs_input.observe(handle_dropdown_change, names='value')


buffer_input = widgets.IntSlider(
    value=5,
    min=0,
    max=100,
    step=1,
    description="Buffer",
    tooltip="Processing Buffer Percentage",
)

# Group B: Output & Storage Settings
outname_input = widgets.Text(
    description="Output Name:", value="globato_dem", placeholder="e.g., newport_dem"
)
outdir_input = widgets.Text(
    description="Out Folder:",
    value="~/workshop/output",
    placeholder="Output directory path",
)
cache_input = widgets.Text(
    description="Cache Dir:",
    value="~/workshop/shared_cache",
    placeholder="Shared cache path",
)

# Group C: Dynamic Sources with Hover Tooltips
source_checkboxes = []
for src_name, src_desc in globato_sources.items():
    cb = widgets.Checkbox(
        value=False,
        description=src_name,
        indent=False,
        tooltip=src_desc,  # <-- Native ipywidgets hover text!
    )
    source_checkboxes.append(cb)

sources_ui = widgets.VBox(
    [widgets.HTML("<b>Data Sources (Hover for info):</b>")] + source_checkboxes,
    layout=widgets.Layout(
        max_height="160px", overflow="auto", border="1px solid #ddd", padding="5px"
    ),
)

# Group D: Build Execution
build_button = widgets.Button(
    description="Build DEM",
    button_style="success",
    icon="play",
    layout=widgets.Layout(margin="10px 0px 0px 0px", width="100%"),
)
output_log = widgets.Output()
fetchez.recipe.setup_logging = lambda *args, **kwargs: None


# --- Logging ---
class OutputWidgetHandler(logging.Handler):
    def __init__(self, widget):
        super().__init__()
        self.widget = widget
        self.setFormatter(
            logging.Formatter("[ %(levelname)s ] %(module)s: %(message)s")
        )

    def emit(self, record):
        # append_stdout pushes the text safely into the widget UI
        self.widget.append_stdout(self.format(record) + "\n")


root_logger = logging.getLogger()
root_logger.setLevel(logging.INFO)


# Prevent duplicate handlers if you run the cell multiple times
if not any(isinstance(h, OutputWidgetHandler) for h in root_logger.handlers):
    root_logger.addHandler(OutputWidgetHandler(output_log))

# --- Execution Buttons ---
cancel_button = widgets.Button(
    description="Cancel Build",
    button_style="danger",
    icon="stop",
    layout=widgets.Layout(margin="10px 0px 0px 0px", width="100%"),
    disabled=True,  # Disabled by default until a build starts
)
clear_button = widgets.Button(
    description="Clear Log",
    button_style="info",
    icon="refresh",
    layout=widgets.Layout(margin="10px 0px 0px 0px", width="100%"),
)


# --- Event Handler: Draw Uploaded Vector ---
def on_upload_change(change):
    if upload_widget.value:
        # Extract file content (ipywidgets 8.x format)
        uploaded_file = upload_widget.value[0]
        file_name = uploaded_file["name"]
        file_content = uploaded_file["content"]

        # 1. Save it locally so Globato can read it
        with open(file_name, "wb") as f:
            f.write(file_content)

        # 2. Update the text box so api.build() uses the file path
        region_input.value = file_name

        # 3. Read with GeoPandas and plot it on the map!
        try:
            gdf = gpd.read_file(file_name)
            geo_layer = GeoData(
                geo_dataframe=gdf,
                style={"color": "black", "fillOpacity": 0.1, "weight": 2},
                name="Vector Upload",
            )
            m.add(geo_layer)
            session_layers.append(geo_layer)

            # Optional: Auto-zoom map to the uploaded vector's bounds
            bounds = gdf.total_bounds  # [minx, miny, maxx, maxy]
            m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])

        except Exception as e:
            with output_log:
                print(f"⚠️ Could not preview vector on map: {e}")


upload_widget.observe(on_upload_change, names="value")


# --- Background Threading for API Execution ---
def run_build_thread(sources, region, increment, buffer, srs, outname, outdir, cache):
    """Runs in the background so the UI doesn't freeze!"""

    outdir = os.path.expanduser(outdir)
    cache = os.path.expanduser(cache)

    fetchez.core.STOP_EVENT.clear()

    completed_batches = []

    with output_log:
        print(f"🚀 Starting background build for {region}...")
        try:
            # Start the generator
            import globato.api

            pipeline_generator = globato.api.build(
                sources,
                region,
                increment,
                extend=f"0:{buffer}",
                t_srs=srs,
                outname=outname,
                outdir=outdir,
                shared_cache=cache,
            )

            # Iterate through the yielded batches
            for tile_data in pipeline_generator:
                config, target_region, batch_name, abs_cache, base_out, tile_dir = (
                    tile_data
                )

                completed_batches.append(batch_name)

                # Extract bounds and draw the blue success polygon!
                if target_region:
                    w, e, s, n = target_region.to_list()
                    completed_poly = Polygon(
                        locations=[(s, w), (n, w), (n, e), (s, e)],
                        color="blue",
                        fill_color="blue",
                        fill_opacity=0.4,
                        name=batch_name,
                    )
                    # Appending to the map is safe from background threads in Jupyter
                    m.add(completed_poly)
                    session_layers.append(completed_poly)  # Track it!

                output_log.clear_output(wait=True)
                recent_tiles = ", ".join(completed_batches[-5:])

                if len(completed_batches) > 5:
                    recent_tiles = f"... {recent_tiles}"

                print(f"🚀 Build running for {region}...")
                print(f"✅ Completed ({len(completed_batches)} tiles): {recent_tiles}")
                print("-" * 50)

            output_log.clear_output(wait=True)
            print("🎉 Entire DEM build process completed successfully!")
            print(f"✅ Total tiles processed: {len(completed_batches)}")

        except Exception as e:
            # Catch the KeyboardInterrupt raised by the STOP_EVENT in core.py
            if "aborted by user" in str(e).lower() or fetchez.core.STOP_EVENT.is_set():
                print("🛑 Pipeline was successfully cancelled.")
            else:
                print(f"❌ Pipeline failed during execution: {e}")

        finally:
            # RESET BUTTON STATES
            build_button.disabled = False
            cancel_button.disabled = True


# --- Event Handlers ---
def on_draw(target, action, geo_json):
    if action == "created":
        coords = geo_json["geometry"]["coordinates"][0]
        lons = [p[0] for p in coords]
        lats = [p[1] for p in coords]
        region_input.value = (
            f"{min(lons):.5f}/{max(lons):.5f}/{min(lats):.5f}/{max(lats):.5f}"
        )


# --- Event Handler: Start Button ---
def on_build_clicked(b):
    output_log.clear_output()
    selected_sources = [cb.description for cb in source_checkboxes if cb.value]

    if not selected_sources:
        with output_log:
            print("⚠️ Please select at least one data source!")
        return

    build_button.disabled = True
    cancel_button.disabled = False

    with output_log:
        print(f"🚀 Triggering Globato build for {region_input.value}...")
        print(f"📦 Sources: {', '.join(selected_sources)}")
        print(f"📁 Saving '{outname_input.value}' to {outdir_input.value}")
        print(f"🗄️ Using Shared Cache: {cache_input.value}")

        # Spin up the background thread to keep the browser responsive
        thread = threading.Thread(
            target=run_build_thread,
            args=(
                selected_sources,
                region_input.value,
                increment_input.value,
                buffer_input.value,
                srs_input.value,
                outname_input.value,
                outdir_input.value,
                cache_input.value,
            ),
        )
        thread.start()


# def on_build_clicked(b):
#     with output_log:
#         output_log.clear_output()
#
#         selected_sources = [cb.description for cb in source_checkboxes if cb.value]
#
#         if not selected_sources:
#             print("⚠️ Please select at least one data source!")
#             return
#
#         print(f"🚀 Triggering Globato build for {region_input.value}...")
#         print(f"📦 Sources: {', '.join(selected_sources)}")
#         print(f"📁 Saving '{outname_input.value}' to {outdir_input.value}")
#         print(f"🗄️ Using Shared Cache: {cache_input.value}")
#
#         # globato.api.build(
#         #     region=region_input.value,
#         #     increment=increment_input.value,
#         #     t_srs=srs_input.value,
#         #     outname=outname_input.value,
#         #     outdir=outdir_input.value,
#         #     shared_cache=cache_input.value,
#         #     sources=selected_sources
#         # )


def on_clear_clicked(b):
    with output_log:
        output_log.clear_output()

    for layer in session_layers:
        if layer in m.layers:
            m.remove(layer)

    session_layers.clear()
    draw_control.clear()


def on_cancel_clicked(b):
    with output_log:
        print(
            "🛑 Cancellation requested! Aborting current tasks (this may take a second)..."
        )

    # Trigger the Fetchez global stop event
    fetchez.core.STOP_EVENT.set()

    # Disable the cancel button to prevent spam-clicking
    cancel_button.disabled = True


draw_control.on_draw(on_draw)
build_button.on_click(on_build_clicked)
cancel_button.on_click(on_cancel_clicked)
clear_button.on_click(on_clear_clicked)


# Assemble the Right Column
form_ui = widgets.VBox(
    [
        widgets.HTML("<b>1. Spatial Configuration</b>"),
        upload_widget,
        region_input,
        increment_input,
        srs_input,
        # other_text,
        buffer_input,
        widgets.HTML("<hr/><b>2. Storage & Outputs</b>"),
        outname_input,
        outdir_input,
        cache_input,
        widgets.HTML("<hr/>"),
        sources_ui,
        widgets.HTML("<hr/>"),
        build_button,
        cancel_button,
        clear_button,
    ]
)


# --- Assemble Dashboard ---
dashboard = widgets.VBox(
    [header, widgets.HBox([m, form_ui]), widgets.HTML("<hr/>"), output_log]
)


display(dashboard)